# Department Performance ETL

## Purpose
Provide real-time department-level performance metrics for category management and business strategy.

## Input
* **Source:** `big_data.silver.order_products` 
* **Source:** `big_data.silver.products_enriched` 
* **Source:** `big_data.silver.orders` 

## Output
* **Target:** `big_data.gold.vw_department_performance`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. JOIN order_products with products_enriched to get department
2. JOIN with orders for order-level data
3. GROUP BY department to calculate total_orders, total_revenue_usd, avg_reorder_rate
4. ORDER BY total_revenue_usd DESC

In [0]:
%sql
-- Department Performance View
-- Purpose: Real-time department-level KPIs for category management

CREATE OR REPLACE VIEW big_data.gold.vw_department_performance AS
SELECT 
  p.department,
  COUNT(DISTINCT o.order_id) AS total_orders,
  ROUND(SUM(p.price_usd), 2) AS total_revenue_usd,
  ROUND(AVG(CASE WHEN op.reordered THEN 1.0 ELSE 0.0 END) * 100, 2) AS avg_reorder_rate
FROM big_data.silver.order_products op
JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
JOIN big_data.silver.orders o ON op.order_id = o.order_id
GROUP BY p.department
ORDER BY total_revenue_usd DESC;

In [0]:
%sql
-- Verify view exists and preview top 5 departments by revenue
-- Returns 21 rows (one per department)

SELECT * FROM big_data.gold.vw_department_performance
LIMIT 5;